# Vanilla Encoder

```python
import torch
import torch.nn as nn

class VanillaEncoder(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers):
        super(VanillaEncoder, self).__init__()
        
        # 1. Embedding Layer: Converts input tokens/features to d_model dimensions
        self.embedding = nn.Linear(input_dim, d_model)
        
        # 2. Define a single Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, # Standard expansion factor
            batch_first=True
        )
        
        # 3. Stack multiple layers
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        # x shape: [batch_size, seq_len, input_dim]
        x = self.embedding(x)
        output = self.transformer_encoder(x)
        return output

# Example Usage:
# batch_size=32, seq_len=10, input_dim=64
# model: d_model=128, 8 attention heads, 6 layers deep
model = VanillaEncoder(input_dim=64, d_model=128, nhead=8, num_layers=6)
sample_input = torch.randn(32, 10, 64)
encoded_output = model(sample_input)

print(f"Output shape: {encoded_output.shape}") 
# Expected: [32, 10, 128]
```

# The Structure of a Vanilla Encoder Layer
A single Encoder layer consists of two main sub-layers. Each sub-layer uses a residual connection (adding the input back to the output) followed by Layer Normalization.
1. **Multi-Head Self-Attention**: This allows the model to focus on different parts of the input sequence simultaneously to understand context (e.g., which "it" refers to in a sentence).
2. **Position-wise Feed-Forward Network (FFN)**: A simple fully connected network applied to each position separately and identically. It usually consists of two linear transformations with a ReLU activation in between.



In [ ]:
import torch
import torch.nn as nn

class ChannelTimeEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=256, dropout=0.1):
        super().__init__()
        # 1. Channel Attention (Inter-channel dependencies)
        self.channel_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm_chan = nn.LayerNorm(d_model)
        
        # 2. Time Attention (Temporal dependencies)
        self.time_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm_time = nn.LayerNorm(d_model)
        
        # 3. Feed Forward Network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm_ff = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Input x shape: [Batch, Num_Patches (N), Num_Channels (M), d_model (D)]
        """
        B, N, M, D = x.shape

        # --- STEP 1: CHANNEL ATTENTION ---
        # Reshape to treat Channels as the sequence: [B * N, M, D]
        chan_in = x.view(B * N, M, D)
        chan_out, _ = self.channel_attn(chan_in, chan_in, chan_in)
        # Residual + Norm
        x = x + self.dropout(chan_out.view(B, N, M, D))
        x = self.norm_chan(x)

        # --- STEP 2: TIME ATTENTION ---
        # Reshape to treat Patches as the sequence: [B * M, N, D]
        # We permute N and M so that N (time) is the sequence length
        time_in = x.transpose(1, 2).reshape(B * M, N, D) 
        time_out, _ = self.time_attn(time_in, time_in, time_in)
        # Reshape back to [B, N, M, D]
        time_out = time_out.view(B, M, N, D).transpose(1, 2)
        
        # Residual + Norm
        x = x + self.dropout(time_out)
        x = self.norm_time(x)

        # --- STEP 3: FEED FORWARD ---
        ff_out = self.ff(x)
        x = self.norm_ff(x + self.dropout(ff_out))
        
        return x